# Chapter 04 - Decorators

#### 1. Mechanical Refresher
A decorator is a function that receives another function and returns a replacement function, and `@decorator_name` above a function is assignment shorthand for `function_name = decorator_name(function_name)`. The wrapper usually accepts `*args` and `**kwargs`, runs code before or after the original function, and returns the original result so the decorated function keeps the same outside behavior.

#### 2. Minimal Working Example

In [ ]:
def announce(func):
    def wrapper(*args, **kwargs):
        print("calling", func.__name__)
        return func(*args, **kwargs)
    return wrapper

@announce
def add(a, b):
    return a + b

print(add(2, 3))

Python first creates `add`, then passes it into `announce`, then stores the returned `wrapper` back under the name `add`. When `add(2, 3)` runs, the wrapper runs first, calls the original `add`, and returns its result.

#### 3. Modify Drills

**Modify Drill 1.** Change the decorator's printed message and predict the output order.

In [ ]:
def trace(func):
    def wrapper(*args, **kwargs):
        print("start")
        result = func(*args, **kwargs)
        print("end")
        return result
    return wrapper

@trace
def double(x):
    return x * 2

actual = double(4)
expected = 8
print("expected:", expected, "actual:", actual, "match:", actual == expected)

**Modify Drill 2.** Use `functools.wraps` and predict the function name before and after removing it.

In [ ]:
from functools import wraps

def preserve_name(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@preserve_name
def score():
    return 1

actual = score.__name__
expected = "score"
print("expected:", expected, "actual:", actual, "match:", actual == expected)

**Modify Drill 3.** Use `@property` syntax now that decorator syntax has been built.

In [ ]:
class Window:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    @property
    def area(self):
        return self.width * self.height

actual = Window(3, 5).area
expected = 15
print("expected:", expected, "actual:", actual, "match:", actual == expected)

#### 4. Break-and-Fix Drills

**Break-and-Fix Drill 1.** Break it by deleting `return wrapper`. Predict why the decorated function becomes `None`, then restore the return.

In [ ]:
def identity_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@identity_decorator
def greet(name):
    return "hi " + name

actual = greet("Ada")
expected = "hi Ada"
print("expected:", expected, "actual:", actual, "match:", actual == expected)

**Break-and-Fix Drill 2.** Break it by deleting `return result`. Predict why callers receive `None`, then restore the return value.

In [ ]:
def noisy(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print("done")
        return result
    return wrapper

@noisy
def triple(x):
    return x * 3

actual = triple(4)
expected = 12
print("expected:", expected, "actual:", actual, "match:", actual == expected)

#### 5. Self-Verification
Expected-vs-actual prints verify the returned values. For decorator control flow, also predict the printed order: wrapper code before the original function runs first, then the original call, then wrapper code after it.

#### 6. Standalone Exercises

**Exercise 1.** Write `add_prefix` so decorated functions return `'prefix:' + original_result`. Expected behavior: `'prefix:ok'`.

In [ ]:
def add_prefix(func):
    def wrapper(*args, **kwargs):
        # TODO: call func and add the prefix.
        return None
    return wrapper

@add_prefix
def status():
    return "ok"

actual = status()
expected = "prefix:ok"
print("expected:", expected, "actual:", actual, "match:", actual == expected)

**Exercise 2.** Write `call_count` so it counts calls in a closure. Expected behavior: `[1, 2]`.

In [ ]:
def call_count(func):
    count = 0
    def wrapper(*args, **kwargs):
        # TODO: update count and return it.
        return None
    return wrapper

@call_count
def ping():
    return "pong"

actual = [ping(), ping()]
expected = [1, 2]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

**Exercise 3.** Use `@staticmethod` in a class. Expected behavior: `[False, True]`.

In [ ]:
class SizeRules:
    # TODO: turn positive into a static method.
    def positive(size):
        return size > 0

rules = SizeRules()
actual = [False, False]  # TODO: after adding @staticmethod, call rules.positive for 0 and 4.
expected = [False, True]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

**Exercise 4.** Use `functools.lru_cache` to cache `fib`. Expected behavior: `8`.

In [ ]:
from functools import lru_cache

# TODO: decorate fib with lru_cache(maxsize=None).
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

actual = fib(6)
expected = 8
print("expected:", expected, "actual:", actual, "match:", actual == expected)

**Exercise 5.** Preserve function metadata with `wraps`. Expected behavior: `actual == 'load_batch'`.

In [ ]:
from functools import wraps

def transparent(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@transparent
def load_batch():
    return [1, 2, 3]

actual = load_batch.__name__
expected = "load_batch"
print("expected:", expected, "actual:", actual, "match:", actual == expected)

#### 7. Applied AI/ML Drill
**ML to Python mirror:** timing or logging a training step is just wrapping a function call and returning the original result. **Python to ML mirror:** training frameworks use the same decorator shape for logging, caching, tracing, or changing call behavior around functions that still look normal to the caller.

**Applied Drill.** Complete `log_loss` so it prints and returns the training-step loss. Expected behavior: returned loss is `0.25`.

In [ ]:
from functools import wraps

def log_loss(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        loss = func(*args, **kwargs)
        # TODO: print the loss and return it.
        return None
    return wrapper

@log_loss
def train_step(weight, x, target):
    prediction = weight * x
    return abs(prediction - target)

actual = train_step(2.0, 3.0, 5.75)
expected = 0.25
print("expected:", expected, "actual:", actual, "match:", actual == expected)

#### 8. Common Bugs
- Returning the original function call at decoration time: the symptom is code running before you call the decorated function.
- Forgetting `return wrapper`: the decorated name becomes `None`.
- Forgetting `return func(...)` inside the wrapper: the original work happens but callers receive `None`.
- Omitting `*args` or `**kwargs`: the decorator only works for one narrow signature.
- Omitting `wraps`: metadata such as `__name__` shows `wrapper`, which makes logs and debugging harder.

#### 9. Compounding Drill

Combine decorators with Chapter 2 methods: write a decorator that counts method calls while still passing `self` through `*args`.

In [ ]:
def count_calls(func):
    count = 0
    def wrapper(*args, **kwargs):
        # TODO: count calls and return original result plus count.
        return None
    return wrapper

class Accumulator:
    def __init__(self):
        self.total = 0

    @count_calls
    def add(self, value):
        self.total = self.total + value
        return self.total

acc = Accumulator()
actual = [acc.add(2), acc.add(3)]
expected = [(2, 1), (5, 2)]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

#### 10. Chapter Project
No chapter project in this chapter. Projects appear every 2-3 chapters so the longer drills can compound several mechanics at once.

#### 11. Solutions Appendix

--- SOLUTIONS: DO NOT PEEK UNTIL ATTEMPTED ---

In [ ]:
# Standalone Exercises 1-2
def add_prefix(func):
    def wrapper(*args, **kwargs):
        return "prefix:" + func(*args, **kwargs)
    return wrapper

@add_prefix
def status():
    return "ok"

print(status())

def call_count(func):
    count = 0
    def wrapper(*args, **kwargs):
        nonlocal count
        count = count + 1
        func(*args, **kwargs)
        return count
    return wrapper

@call_count
def ping():
    return "pong"

print([ping(), ping()])

In [ ]:
# Standalone Exercises 3-5
from functools import lru_cache, wraps

class SizeRules:
    @staticmethod
    def positive(size):
        return size > 0

print([SizeRules.positive(0), SizeRules.positive(4)])

@lru_cache(maxsize=None)
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print(fib(6))

def transparent(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@transparent
def load_batch():
    return [1, 2, 3]

print(load_batch.__name__)

In [ ]:
# Applied AI/ML Drill and Compounding Drill
from functools import wraps

def log_loss(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        loss = func(*args, **kwargs)
        print("loss:", loss)
        return loss
    return wrapper

@log_loss
def train_step(weight, x, target):
    prediction = weight * x
    return abs(prediction - target)

print(train_step(2.0, 3.0, 5.75))

def count_calls(func):
    count = 0
    def wrapper(*args, **kwargs):
        nonlocal count
        count = count + 1
        result = func(*args, **kwargs)
        return result, count
    return wrapper

class Accumulator:
    def __init__(self):
        self.total = 0

    @count_calls
    def add(self, value):
        self.total = self.total + value
        return self.total

acc = Accumulator()
print([acc.add(2), acc.add(3)])